## Base Overlay

In [1]:
import pynq
import numpy as np
import asyncio
import time
from pynq.overlays.base import BaseOverlay

# Make sure "base.bit" and "base.hwh" are in the same directory
print("Loading Base Overlay...")
ol = BaseOverlay("base.bit")
print("Base Overlay loaded successfully!\n")

print("=== Base Overlay Metadata ===")
print(f"Bitstream Path : {ol.bitfile_name}")
print(f"Timestamp      : {ol.timestamp}")

print("\n=== Available IPs (ip_dict) ===")
for ip_name in ol.ip_dict.keys():
    print(f"  - {ip_name}")

print("\n=== Available Interrupts (interrupt_pins) ===")
for intr_name in ol.interrupt_pins.keys():
    print(f"  - {intr_name}")

print("\n=== Driver Binding Verification ===")
# Verify that the custom AxiTimer driver was automatically bound
print(f"Timer IP Class : {type(ol.timer)}")

/usr/local/share/pynq-venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)
/usr/local/share/pynq-venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)


Loading Base Overlay...


Base Overlay loaded successfully!

=== Base Overlay Metadata ===
Bitstream Path : /usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/base/base.bit
Timestamp      : 2026/4/21 20:48:40 +727109

=== Available IPs (ip_dict) ===
  - led_pl_top/axi_gpio_led_pl
  - intc_top/axi_intc_0
  - intc_top/axi_gpio_int_trig_in0
  - dma_loopback_top/axi_dma_0
  - intc_top/axi_timer_0
  - dma_loopback_top/axis_fifo_data_counts
  - userio_top/axi_gpio_userio
  - fiberio_top/axi_gpio_fiberio
  - bin_counter_top/axi_gpio_bin_cnt
  - debug_bridge_0
  - ps_top/zynq_ultra_ps_e_0

=== Available Interrupts (interrupt_pins) ===
  - intc_top/axi_gpio_int_trig_in0/TRI_O
  - intc_top/intr_concat/In0
  - dma_loopback_top/axi_dma_0/mm2s_introut
  - intc_top/intr_concat/In1
  - dma_loopback_top/axi_dma_0/s2mm_introut
  - intc_top/intr_concat/In2
  - intc_top/axi_timer_0/interrupt
  - intc_top/intr_concat/In3
  - intc_top/xlconstant_0/dout
  - intc_top/intr_concat/In4
  - intc_top/intr_concat/In5
  -

## GPIO Test

In [2]:
# ==========================================
# 1. Test LED
# ==========================================
print("--- Testing LED ---")
ol.write_led(1)
print("LED is ON. Check your board.")
time.sleep(1)
ol.write_led(0)
print("LED is OFF.")

# ==========================================
# 2. Test Binary Counter (32-bit)
# ==========================================
print("\n--- Testing Binary Counter ---")
count1 = ol.read_binary_counter()
time.sleep(0.5)
count2 = ol.read_binary_counter()
print(f"Counter Read 1: {count1}")
print(f"Counter Read 2: {count2}")
if count2 is not None and count1 is not None and count2 > count1:
    print("Binary counter is ticking correctly!")

# ==========================================
# 3. Test User IO (4-bit)
# ==========================================
print("\n--- Testing User IO (4-bit) ---")
# Writing a 4-bit value (e.g., 0xA = 1010 in binary)
test_val_user = 0xA
ol.write_userio(test_val_user)
print(f"Wrote {hex(test_val_user)} to User IO Output (Ch2).")
# Note: Unless you physically loop back User IO output pins to input pins,
# reading from Ch1 will likely just return floating/default states.
user_in = ol.read_userio()
print(f"Read {hex(user_in) if user_in is not None else 'None'} from User IO Input (Ch1).")

# ==========================================
# 4. Test Fiber IO (2-bit)
# ==========================================
print("\n--- Testing Fiber IO (2-bit) ---")
# Writing a 2-bit value (e.g., 0x3 = 11 in binary)
test_val_fiber = 0x3
ol.write_fiberio(test_val_fiber)
print(f"Wrote {hex(test_val_fiber)} to Fiber IO Output (Ch2).")
fiber_in = ol.read_fiberio()
print(f"Read {hex(fiber_in) if fiber_in is not None else 'None'} from Fiber IO Input (Ch1).")

--- Testing LED ---
LED is ON. Check your board.
LED is OFF.

--- Testing Binary Counter ---
Counter Read 1: 793145757
Counter Read 2: 843296926
Binary counter is ticking correctly!

--- Testing User IO (4-bit) ---
Wrote 0xa to User IO Output (Ch2).
Read 0x0 from User IO Input (Ch1).

--- Testing Fiber IO (2-bit) ---
Wrote 0x3 to Fiber IO Output (Ch2).
Read 0x3 from Fiber IO Input (Ch1).


## AXI Interrupt Controller

### AXI Timer Interrupt

In [3]:
async def test_timer_interrupt():
    print("--- Testing AXI Timer & Interrupts ---")
    
    # 1. Check current interrupt status
    print("\n[Pre-Test] Interrupt Counters:")
    ol.check_interrupts(filter_keyword="fabric")
    
    # 2. Start timer for 2 seconds
    delay = 2.0
    print(f"\nStarting AXI Timer for {delay} seconds...")
    ol.timer.start(delay_sec=delay)
    
    # 3. Wait for the hardware interrupt asynchronously
    start_time = time.time()
    await ol.timer.wait_async()
    end_time = time.time()
    
    # 4. Clear the interrupt flag
    ol.timer.clear_interrupt()
    
    print(f"Timer Interrupt caught! Elapsed time: {end_time - start_time:.2f} seconds.")
    
    # 5. Check interrupt status again (count should increment)
    print("\n[Post-Test] Interrupt Counters:")
    ol.check_interrupts(filter_keyword="fabric")

# Execute the async function
await test_timer_interrupt()

--- Testing AXI Timer & Interrupts ---

[Pre-Test] Interrupt Counters:
=== Interrupt Status (Filter: 'fabric') ===
CPU0       CPU1       CPU2       CPU3
49:          3          0          0          0     GICv2 121 Level     fabric

Starting AXI Timer for 2.0 seconds...
Timer Interrupt caught! Elapsed time: 2.00 seconds.

[Post-Test] Interrupt Counters:
=== Interrupt Status (Filter: 'fabric') ===
CPU0       CPU1       CPU2       CPU3
49:          4          0          0          0     GICv2 121 Level     fabric


### AXI GPIO Interrupt

In [4]:
async def test_gpio_interrupt():
    print("--- Testing GPIO Triggered Interrupt 0 ---")
    
    # Create a background task to listen for the interrupt
    # This simulates a responsive system waiting for external events
    listen_task = asyncio.create_task(ol.wait_gpio_interrupt())
    
    print("Listening for GPIO interrupt on 'intr_concat/In0'...")
    await asyncio.sleep(1) # Simulate doing other work
    
    print("Triggering hardware interrupt via GPIO pulse...")
    ol.trigger_gpio_interrupt()
    
    # Wait for the listener task to confirm reception
    await listen_task
    print("GPIO Interrupt successfully captured!")

# Execute the async function
await test_gpio_interrupt()

--- Testing GPIO Triggered Interrupt 0 ---
Listening for GPIO interrupt on 'intr_concat/In0'...
Triggering hardware interrupt via GPIO pulse...
GPIO Interrupt successfully captured!


## DMA

In [5]:
async def test_dma_loopback():
    print("--- Testing DMA Async Loopback ---")
    
    print("\n[Pre-Test] CMA Memory Status:")
    ol.check_cma_info()
    
    # Allocate 10MB of contiguous memory for input and output
    # 2,500,000 uint32 elements = 10,000,000 bytes
    num_elements = 2500000
    print(f"\nAllocating {num_elements * 4 / (1024*1024):.2f} MB of CMA memory for buffers...")
    input_buffer = pynq.allocate(shape=(num_elements,), dtype=np.uint32)
    output_buffer = pynq.allocate(shape=(num_elements,), dtype=np.uint32)
    
    # Fill input buffer with sequential data
    input_buffer[:] = np.arange(num_elements, dtype=np.uint32)
    
    print("\n[Mid-Test] CMA Memory Status (Notice the drop in CmaFree):")
    ol.check_cma_info()
    
    # Perform asynchronous DMA transfer
    print("\nInitiating DMA transfer (MM2S -> S2MM loopback)...")
    start_time = time.time()
    success = await ol.dma_transfer_async(input_buffer, output_buffer)
    end_time = time.time()
    
    if success:
        print(f"DMA Transfer completed in {end_time - start_time:.4f} seconds.")
        # Verify data integrity
        if np.array_equal(input_buffer, output_buffer):
            print("SUCCESS: Output data perfectly matches input data!")
        else:
            print("ERROR: Data mismatch detected!")
    else:
        print("DMA Transfer failed.")
        
    # Free the allocated CMA memory
    print("\nFreeing buffers...")
    del input_buffer
    del output_buffer
    
    print("\n[Post-Test] CMA Memory Status (CmaFree should recover):")
    ol.check_cma_info()

# Execute the async function
await test_dma_loopback()

--- Testing DMA Async Loopback ---

[Pre-Test] CMA Memory Status:
=== CMA Memory Status (/proc/meminfo) ===
CmaTotal:         262144 kB
CmaFree:          257628 kB

Allocating 9.54 MB of CMA memory for buffers...

[Mid-Test] CMA Memory Status (Notice the drop in CmaFree):
=== CMA Memory Status (/proc/meminfo) ===
CmaTotal:         262144 kB
CmaFree:          238448 kB

Initiating DMA transfer (MM2S -> S2MM loopback)...
DMA Transfer completed in 0.0082 seconds.
SUCCESS: Output data perfectly matches input data!

Freeing buffers...

[Post-Test] CMA Memory Status (CmaFree should recover):
=== CMA Memory Status (/proc/meminfo) ===
CmaTotal:         262144 kB
CmaFree:          256428 kB


In [6]:
async def run_dma_throughput_benchmark(target_size_mb=32):
    """
    Benchmarks the DMA performance by transferring a specific amount of data 
    through the loopback path.
    """
    print(f"--- DMA Throughput Benchmark ---")
    
    # Calculate the number of uint32 elements (4 bytes per element)
    # 32 MB = 8,388,608 elements
    num_elements = (target_size_mb * 1024 * 1024) // 4
    actual_data_size_mb = (num_elements * 4) / (1024 * 1024)

    print(f"Status: Allocating {actual_data_size_mb:.2f} MB of CMA memory...")
    
    # Allocate contiguous memory buffers
    input_buffer = pynq.allocate(shape=(num_elements,), dtype=np.uint32)
    output_buffer = pynq.allocate(shape=(num_elements,), dtype=np.uint32)
    
    # Initialize input buffer with a pattern
    input_buffer[:] = np.arange(num_elements, dtype=np.uint32)

    print(f"Status: Initiating asynchronous transfer...")
    
    # Record start time
    start_time = time.perf_counter()
    
    # Perform the DMA transfer using the custom BaseOverlay API
    # This involves both MM2S and S2MM channels
    success = await ol.dma_transfer_async(input_buffer, output_buffer)
    
    # Record end time
    end_time = time.perf_counter()
    
    duration = end_time - start_time

    if success:
        # Calculate throughput: (Size in MB) / (Time in seconds)
        throughput = actual_data_size_mb / duration
        print(f"\nResults:")
        print(f" - Payload Size : {actual_data_size_mb:.2f} MB")
        print(f" - Elapsed Time : {duration:.6f} seconds")
        print(f" - Throughput   : {throughput:.2f} MB/s")
        
        # Verify data integrity
        if np.array_equal(input_buffer, output_buffer):
            print("\nVerification: SUCCESS (Data integrity confirmed)")
        else:
            print("\nVerification: FAILED (Data mismatch detected)")
    else:
        print("\nError: DMA transfer failed to complete.")

    # Explicitly free the CMA memory
    del input_buffer
    del output_buffer

# Execute the benchmark
await run_dma_throughput_benchmark()

--- DMA Throughput Benchmark ---
Status: Allocating 32.00 MB of CMA memory...
Status: Initiating asynchronous transfer...

Results:
 - Payload Size : 32.00 MB
 - Elapsed Time : 0.022870 seconds
 - Throughput   : 1399.18 MB/s

Verification: SUCCESS (Data integrity confirmed)
